# Load Packages

The following cell loads the necessary packages to run ezTrackRT. It does not need to be changed.

In [ ]:
%load_ext autoreload
%autoreload 2

from matplotlib import pyplot as plt
import RT_functions as rt

# Start the Video Stream

The following cell connects ezTrackRT to the video camera and starts the video stream. The default camera (`src=0`) is used. This cell does not need to be changed unless a different camera is being used.

In [ ]:
vid = rt.Video(src=0)
vid.start()

# Define Cropping Bounds

The following cell allows you to define a portion of the video frame that will be used for location tracking and/or freezing analysis. Use the box selection tool (square with a + sign) to draw a rectangle around the area you want to keep. Double-click to define one corner of the area to be cropped, and double-click again to define the opposite corner. The selected region will be used to crop each incoming frame before it is processed for location tracking and/or freezing analysis. This step is optional.

In [ ]:
%output size = 100

vid.crop_define()

# Define Distance Scale

The following cell allows you to define a physical distance scale that can be used to report distances during location tracking. It is also applicable when using the distance freezing method. Enter a name for the desired distance unit (e.g., `'cm'`) and the known distance between two points in that unit. Use the point selection tool (the icon with three circles) to mark the two corresponding points on the video frame. Click once to place the first point, and click again to place the second point. Here, `name = 'cm'` and `dist = 10` are examples indicating that the distance between the two selected points is 10 cm; these values should be changed to match the physical reference being used. This step is only needed if you would like distances to be reported in a physical unit rather than pixels.

In [ ]:
%output size = 100

vid.distance_define(name = 'cm', dist = 10)

# Select Regions to Exclude from Tracking

The following cell allows you to define one or more regions of the video frame to exclude from location tracking and/or freezing analysis. Use the polygon drawing tool (the polygon-shaped icon) to outline the region(s) you want to exclude. Double-click to start and finish a region, and single-click to add additional vertices to the region. Excluded regions will not be considered when ezTrackRT determines the animal's location and/or freezing behavior. This step is optional.

In [ ]:
%output size = 100

vid.mask_define()

# Select Regions of Interest

The following cell allows you to define one or more regions of interest (ROIs) within the video frame. Enter the desired ROI name(s) (e.g., `names = ['left']`). For multiple ROIs, separate them using commas and ensure each name is placed in quotation marks. Use the polygon drawing tool (the polygon-shaped icon) to outline each ROI. Double-click to start and finish a region, and single-click to add additional vertices. ROIs can be used to generate ROI-specific data (e.g., how much time the animal spends in ROI 1 vs. ROI 2). Note that ROIs may only be used with location tracking and the distance freezing method.

In [ ]:
%output size = 100

vid.roi_define(names = ['left'])

# Save cropping/roi/masking parameters

The following cell saves the cropping, ROI, masking, and distance-scale settings. The saved file is a `.pickle` file and should be given a descriptive filename. Set `filename` to the full path where you want to save the file, including the desired filename at the end of the path. Include the `.pickle` extension in the filename. The folder structure where you want to save the file must already exist; `params_save()` does not create new folders.

**Windows:** Use a path such as:
`filename = r'C:\Users\YourName\Documents\ezTrackRT\parameters.pickle'`

**macOS:** Use a path such as:
`filename = '/Users/YourName/Documents/ezTrackRT/parameters.pickle'`

**Linux:** Use a path such as:
`filename = '/home/YourName/Documents/ezTrackRT/parameters.pickle'`

In [ ]:
filename = r'C:\Users\mcter\OneDrive\Documents\ZP_Lab\ezTrack\RealTime\Test\parameters.pickle'

vid.params_save(file = filename)

# Load cropping/roi/masking parameters

The following cell loads previously saved cropping, ROI, masking, and distance-scale settings. Change `filename` to the full path of the `.pickle` parameters file you previously saved. The file must already exist at the specified location, and the path should include the filename at the end.

**Windows:** Use a path such as:
`filename = r'C:\Users\YourName\Documents\ezTrackRT\parameters.pickle'`

**macOS:** Use a path such as:
`filename = '/Users/YourName/Documents/ezTrackRT/parameters.pickle'`

**Linux:** Use a path such as:
`filename = '/home/YourName/Documents/ezTrackRT/parameters.pickle'`

In [ ]:
filename = r'C:\Users\mcter\OneDrive\Documents\ZP_Lab\ezTrack\RealTime\Test\parameters.pickle'

vid.params_load(file = filename)

# Set Reference Frame for Tracking

The following cell creates a reference frame that is used for location tracking and the distance freezing method. While the video is running, `vid.ref_create()` collects frames for the specified duration and calculates their average to create a reference frame representing the background of the recording area. Here, `secs = 1` means that frames collected over 1 second are used to create the reference frame. Adjust this value as needed. `print_sts = False` prevents the progress of reference-frame creation from being printed. The second line displays the resulting reference frame. Check that the reference frame accurately represents the background of the recording area and does not contain the animal or other objects that should be excluded from the background. Note that a reference frame is not required for the pixel freezing method.

In [ ]:
vid.ref_create(secs = 1, print_sts = False)

plt.imshow(vid.ref, cmap='gray')

# Set Analysis-Specific Parameters

**The following cell sets parameters used for location tracking and the distance freezing method.** These settings determine how the animal is identified and localized within each video frame and should be adjusted based on your setup.

* `track_method` determines how differences between the current frame and the reference frame are calculated. Set to `'abs'` to detect changes regardless of whether the animal is lighter or darker than the background. Alternatively, use `'light'` if the animal is lighter than the background or `'dark'` if the animal is darker than the background.
* `track_thresh` determines the percentile threshold used to set smaller pixel differences between the current frame and reference frame to zero before calculating the animal's location. For example, `track_thresh = 95` sets difference values below the 95th percentile to zero.
* `track_window_use` determines whether a window around the animal's previous location is used to give greater weight to that area when determining its current location.
* `track_window_sz` sets the width and height, in pixels, of the square tracking window when `track_window_use = True`.
* `track_window_wt` determines how strongly the tracking window is weighted, from 0 to 1. This parameter only affects tracking when `track_window_use = True`.
* `track_rmvwire` determines whether morphological processing is used to reduce the effect of wires or similar thin structures on location tracking.
* `track_rmvwire_krn` sets the size of the morphological kernel used for wire removal when `track_rmvwire = True`. The kernel should be larger than the wire but smaller than the animal.

In [ ]:
vid.track_method = 'abs'
vid.track_thresh = 95
vid.track_window_use = False
vid.track_window_sz = 100
vid.track_window_wt = 0.9
vid.track_rmvwire = False
vid.track_rmvwire_krn = 10

**The following cell sets parameters used for both the distance and pixel freezing methods.** These settings determine how motion values are collected and used to classify the animal's freezing state and should be adjusted based on your setup.

* `freeze_method` determines the method used to calculate freezing. Set to `'distance'` to calculate freezing based on the animal's center-of-mass movement (similar to the location tracking algorithm) or `'pixel'` to calculate freezing based on pixel fluctuations between consecutive frames. If only location tracking is desired, `freeze_method` can safely be set to either option. Location tracking can be performed independently of freezing analysis.
* `freeze_mtbuffer_sz` determines the number of consecutive frames for which motion values are stored in a buffer. The values represent center-of-mass movement when using the distance method or pixel fluctuations when using the pixel method. These values are used to determine whether the animal is currently freezing.
* `freeze_mtthresh_method` determines how the values stored in the buffer are summarized before being compared with `freeze_thresh`. Set to `'max'` to use the largest value in the buffer or `'mean'` to use the mean value of the buffer.
* `freeze_thresh` sets the motion threshold used to determine whether the animal is freezing. The summarized value based on `freeze_mtthresh_method` is compared with this threshold; the animal is classified as freezing when the summarized value is less than `freeze_thresh`.
* `freeze_mtthresh_thresh` sets the minimum pixel-intensity difference threshold between consecutive frames that is considered to represent movement. Pixel-intensity differences below this value are treated as background noise rather than movement. This parameter only applies to the pixel freezing method.

In [ ]:
vid.freeze_method = 'distance'
vid.freeze_mtbuffer_sz = 15
vid.freeze_mtthresh_method = 'mean'
vid.freeze_thresh = 10
vid.freeze_mtthresh_thresh = 15

# Set Desired Analysis Type

Select whether location tracking and/or freezing analysis is desired.

In [ ]:
vid.track_freeze_set(track = True, freeze = False)

# Initialize Data Writer

The following cell initializes the data writer, which prepares ezTrackRT to save the recorded video and tracking/freezing data. `dpath` specifies the folder where the output files will be saved. The writer is initialized but does not begin saving until `vid.writer_start()` is called. Change `dpath` to reflect the full path to the desired output folder on your computer, which must already exist. `dfilename` specifies the desired name of the saved tracking/freezing data file and must end in `.csv`, while `vfilename` specifies the desired name of the saved video file and must end in `.avi`. Run this cell after completing any cropping, because the data writer uses the dimensions of the current frame when initializing the video file.

**Windows:** Use a path such as:
`dpath = r'C:\Users\YourName\Documents\ezTrackRT\Data'`

**macOS:** Use a path such as:
`dpath = '/Users/YourName/Documents/ezTrackRT/Data'`

**Linux:** Use a path such as:
`dpath = '/home/YourName/Documents/ezTrackRT/Data'`

In [ ]:
vid.writer_init(
    dpath = r'C:\Users\mcter\OneDrive\Documents\ZP_Lab\ezTrack\RealTime\Test',
    dfilename = 'data.csv',
    vfilename = 'video.avi'
)

# Start Data Writer

The following cell signals the data writer to begin saving the recorded video and tracking/freezing data. The data writer must first be initialized using `vid.writer_init()`. Once started, data will be saved to the output directory specified by `dpath` in `vid.writer_init()`. `vid.track_window_reset = True` resets the tracking window before the next frame (if `vid.track_window_use = True`), allowing the window to be repositioned based on the animal's current location. `freeze_mtbuffer_reset = True` clears the motion buffer before the next frame, preventing motion values that were based on previous frames from being used to determine the animal's current freezing state.

In [ ]:
vid.track_window_reset = True
vid.freeze_mtbuffer_reset = True
vid.writer_start()

# Start Video Display.  Press 'q' to quit. 

The following cell starts the video display, which may be stopped by pressing `q`. Note that frame acquisition will continue until `vid.release()` is called in the final cell.

* `show_xy = True` displays the animal's tracked center-of-mass position when location tracking is enabled. This setting also applies when using the distance freezing method.
* `show_dif = True` displays the difference image used for location tracking and/or freezing analysis. For location tracking and the distance freezing method, this is the pixel-wise difference in intensity between the current frame and the reference frame. For the pixel freezing method, this is the pixel-wise difference in intensity between the current frame and the previous frame.
* `show_fz = True` displays the animal's current freezing state (`Freezing` or `Moving`) on the video when freezing analysis is enabled.

In [ ]:
vid.display(show_xy = True, show_dif = False, show_fz = True)

# Stop Saving Video and Release Camera

The following cell stops video processing, stops the data writer, and releases the camera. Run this cell when the experiment is finished. This should be the final cell run before closing the notebook or reconnecting to a different camera. See `vid.stop` and `vid.writer_stop` for more nuanced control.

In [ ]:
vid.release()